In [ ]:
# ### DISCRETIZATION OF GROUND TRUTH IMAGE TO SIMULATE LINEAR TILT CONTINUOUS PSF
# ### NO OVERLAPPING GRID

# # generate ground truth image, "captured image"

# np.random.seed(0)

# # we're guessing this value
# defocus = WAVELENGTH * 2


# image = torch.tensor(
#     np.random.choice(
#         [0, 1],
#         size=(IMAGE_SIZE, IMAGE_SIZE),
#         p=[1 - EMITTER_DENSITY, EMITTER_DENSITY],
#     )
# ).float()

# # brute force continuous PSF using grid-wise discretization to model linear tilt of 2D material
# h, w = image.shape
# NUM_GRID_ACROSS = 4
# NUM_GRIDS = NUM_GRID_ACROSS * NUM_GRID_ACROSS
# GRID_SIZE = h // NUM_GRID_ACROSS

# # GRID LABELS/INDICES
# # | 0 | 1 | 2 | 3 |
# # | 4 | 5 | 6 | 7 |
# # | 8 | 9 | 10| 11|
# # | 12| 13| 14| 15|

# # generate list of psf for NUM_GRIDS
# psfs = []
# temp_defocus = defocus
# for i in range(NUM_GRIDS):
#     temp_defocus = temp_defocus * 1.05
#     temp_psf = (
#         torch.nn.functional.conv2d(
#             microscope_psf(temp_defocus, KERNEL_SIZE).unsqueeze(0).unsqueeze(0),
#             laser_psf(temp_defocus, KERNEL_SIZE).unsqueeze(0).unsqueeze(0),
#             padding="same",
#         )
#         .squeeze(0)
#         .squeeze(0)
#     )
#     psfs.append(temp_psf) # store for reference later

# captured_image = torch.tensor(np.zeros_like(image, dtype=float))
# for i in range(NUM_GRID_ACROSS):
#     for j in range(NUM_GRID_ACROSS):
#         start_y = i * GRID_SIZE
#         end_y = (i+1) * GRID_SIZE
#         start_x = j * GRID_SIZE
#         end_x = (j+1) * GRID_SIZE

#         region = image[start_y:end_y, start_x:end_x]
#         psf_index = (i * NUM_GRID_ACROSS) + j
#         cur_psf = psfs[psf_index]
#         c_region = torch.nn.functional.conv2d(
#                     region.unsqueeze(0).unsqueeze(0), cur_psf.unsqueeze(0).unsqueeze(0), padding="same"
#                     ).squeeze(0).squeeze(0) + NOISE * np.random.uniform(0, 1, size=(GRID_SIZE, GRID_SIZE))
        
#         captured_image[start_y:end_y, start_x:end_x] = c_region

# plt.figure(figsize=(15, 5))
# plt.subplot(2, 3, 1)
# plt.title('Ground Truth Image')
# plt.imshow(image)
# plt.subplot(2, 3, 2)
# plt.title('PSF 0')
# plt.imshow(psfs[0])
# plt.subplot(2, 3, 3)
# plt.title('PSF 1')
# plt.imshow(psfs[1])
# plt.subplot(2, 3, 4)
# plt.title('PSF 2')
# plt.imshow(psfs[2])
# plt.subplot(2, 3, 5)
# plt.title('PSF 3')
# plt.imshow(psfs[3])
# plt.subplot(2, 3, 6)
# plt.title('Noisy Measurement Image') # with Linear Tilt on Continuous PSF
# plt.imshow(captured_image)
# plt.tight_layout()